# Show Reel — Community Persona Identification Pipeline

**Platform:** Instagram  
**Runtime:** Google Colab Enterprise + Vertex AI (Gemini)  
**Data source:** outputs of `Data_Preparation_Pipeline.ipynb`

Pipeline stages:
- **Stage 0** — Feature engineering (user-level aggregation from pre-processed comments)
- **Stage 1** — Exploratory taxonomy discovery (Gemini Flash, ~500 users sample)
- **Stage 2** — Deterministic CAG classification (Gemini Pro, full dataset)

Set `PIPELINE_MODE` in the Configuration cell before running.

## Install Dependencies

Run once per runtime, then restart the kernel.

In [ ]:
!pip install -q google-cloud-aiplatform pandas pyarrow tqdm emoji

## Configuration

Toggle models, pipeline mode, and paths here — no other cell needs editing.

In [ ]:
import os

# GCP / Vertex AI
GCP_PROJECT_ID = "gen-lang-client-0792749758"
GCP_LOCATION   = "us-central1"
GCS_BUCKET     = "afb_showreel"

# Models
# Stage 1 uses Flash (cheaper, fast for taxonomy discovery on a sample).
# Stage 2 uses Pro (higher reasoning quality for full-dataset classification).
MODEL_STAGE1_EXPLORATORY = "gemini-2.5-flash"
MODEL_STAGE2_CLASSIFY    = "gemini-2.5-pro"

# Determinism (mandatory for academic reproducibility)
TEMPERATURE              = 0.0
TOP_P                    = 1.0
MAX_OUTPUT_TOKENS_STAGE1 = 2048
MAX_OUTPUT_TOKENS_STAGE2 = 512

# Pipeline mode
# "SAMPLE" → Stage 0 + Stage 1 (taxonomy discovery, human-in-the-loop)
# "FULL"   → Stage 0 + Stage 2 (production classification, requires approved taxonomy)
# "ALL"    → Both stages in sequence
PIPELINE_MODE = "SAMPLE"

# Sampling
SAMPLE_N_USERS    = 500
SAMPLE_SEED       = 42
BATCH_SIZE_STAGE1 = 20
BATCH_SIZE_STAGE2 = 10

# Artifact paths (local; uploaded to GCS after the pipeline completes)
TAXONOMY_JSON_PATH = "outputs/taxonomy.json"
RESULTS_PATH       = "outputs/user_personas.parquet"
os.makedirs("outputs", exist_ok=True)

print("\u2705 Configuration loaded.")
print(f"   Mode:    {PIPELINE_MODE}")
print(f"   Bucket:  gs://{GCS_BUCKET}")

## Vertex AI Initialization

In [ ]:
import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig

vertexai.init(project=GCP_PROJECT_ID, location=GCP_LOCATION)

def get_model(model_name: str) -> GenerativeModel:
    return GenerativeModel(model_name)

def get_generation_config(max_tokens: int) -> GenerationConfig:
    return GenerationConfig(
        temperature=TEMPERATURE,
        top_p=TOP_P,
        max_output_tokens=max_tokens,
        response_mime_type="application/json",
    )

print("\u2705 Vertex AI initialized.")
print(f"   Project: {GCP_PROJECT_ID} | Region: {GCP_LOCATION}")

### Optional — Model Connectivity Test

Run to verify both models are reachable before starting the pipeline.

In [ ]:
test_prompt = "What is the capital of France?"

for stage, model_name, max_tok in [
    ("Stage 1", MODEL_STAGE1_EXPLORATORY, MAX_OUTPUT_TOKENS_STAGE1),
    ("Stage 2", MODEL_STAGE2_CLASSIFY,    MAX_OUTPUT_TOKENS_STAGE2),
]:
    try:
        resp = get_model(model_name).generate_content(
            test_prompt, generation_config=get_generation_config(max_tok)
        )
        print(f"\u2705 {stage} ({model_name}): {resp.text[:60]}")
    except Exception as e:
        print(f"\u274c {stage} ({model_name}): {e}")

## Data Loading

Reads the three files produced by `Data_Preparation_Pipeline.ipynb` plus the
raw post-level metadata (not yet covered by the pipeline).

| File | Source | Description |
|---|---|---|
| `comments_ml.parquet` | pipeline | Pre-computed comment-level features |
| `comments_gml.parquet` | pipeline | Graph edge list (used for reply flag) |
| `ig_posts_cleaned.parquet` | raw | Post metadata for temporal features |

In [ ]:
import pandas as pd

base_path = f"gs://{GCS_BUCKET}/"
print(f"Accessing bucket: {base_path}")

print("Reading comments_ml.parquet  ...")
ig_comments_ml  = pd.read_parquet(base_path + "comments_ml_instagram.parquet")

print("Reading comments_gml.parquet ...")
ig_comments_gml = pd.read_parquet(base_path + "comments_gml_instagram.parquet")

print("Reading ig_posts_cleaned.parquet ...")
ig_media = pd.read_parquet(base_path + "ig_posts_cleaned.parquet")

print(f"\n\u2705 Datasets loaded.")
print(f"   comments_ml:  {len(ig_comments_ml):,} rows")
print(f"   comments_gml: {len(ig_comments_gml):,} rows")
print(f"   ig_media:     {len(ig_media):,} rows")

# Merge reply_to_comment_id from the GML frame into the ML frame
ig_comments = ig_comments_ml.merge(
    ig_comments_gml[["comment_id", "reply_to_comment_id"]],
    on="comment_id", how="left"
)

# Validation
n_before    = len(ig_comments)
ig_comments = ig_comments[ig_comments["author_id"].notna()].copy()
ig_media    = ig_media[ig_media["media_id"].notna()].copy()

print(f"\nPost-Validation Summary:")
print(f"   Comments retained: {len(ig_comments):,} / {n_before:,}")
print(f"   Unique commenters: {ig_comments['author_id'].nunique():,}")
print(f"   Unique media IDs:  {ig_media['media_id'].nunique():,}")

display(ig_comments.head(2))
display(ig_media.head(2))

## Stage 0 — Feature Engineering

All comment-level features (word count, emoji count, etc.) are pre-computed by
the Data Preparation Pipeline and live in `comments_ml.parquet`.

This stage:
1. Derives binary flags from the pre-computed counts
2. Merges with post metadata for temporal features (`hours_to_comment`)
3. Aggregates to a **user-level feature matrix**
4. Attaches a representative comment sample from `comments_llm.jsonl`

In [ ]:
from datetime import timedelta

# 1. Binary flags from pre-computed counts
ig_comments["has_emoji"]    = (ig_comments["emoji_count"] > 0).astype(int)
ig_comments["has_question"] = (ig_comments["question_count"] > 0).astype(int)
ig_comments["has_exclaim"]  = (ig_comments["exclamation_count"] > 0).astype(int)
ig_comments["is_reply"]     = ig_comments["reply_to_comment_id"].notna().astype(int)
print("\u2705 Binary flags derived.")

# 2. Merge with post metadata for temporal features
media_cols = ["media_id", "timestamp", "like_count", "comments_count",
              "media_product_type", "media_type", "engagement_rate"]
existing_media_cols = [c for c in media_cols if c in ig_media.columns]

ig_comments = ig_comments.merge(
    ig_media[existing_media_cols].rename(columns={
        "timestamp":      "post_timestamp",
        "like_count":     "post_like_count",
        "comments_count": "post_comments_count",
    }),
    on="media_id", how="left"
)

ig_comments["timestamp"]      = pd.to_datetime(ig_comments["timestamp"])
ig_comments["post_timestamp"] = pd.to_datetime(ig_comments["post_timestamp"])
ig_comments["hours_to_comment"] = (
    (ig_comments["timestamp"] - ig_comments["post_timestamp"])
    .dt.total_seconds() / 3600
).clip(lower=0)

# 3. User-level aggregation
def build_user_feature_matrix(df: pd.DataFrame) -> pd.DataFrame:
    grp = df.groupby("author_id")
    agg = {
        "total_comments":          grp.size(),
        "unique_posts_commented":  grp["media_id"].nunique(),
        "total_replies_made":      grp["is_reply"].sum(),
        "reply_ratio":             grp["is_reply"].mean(),
        "mean_hours_to_comment":   grp["hours_to_comment"].mean(),
        "median_hours_to_comment": grp["hours_to_comment"].median(),
        "pct_comments_under_1h":   grp["hours_to_comment"].apply(lambda x: (x < 1).mean()),
        "pct_comments_under_24h":  grp["hours_to_comment"].apply(lambda x: (x < 24).mean()),
        "activity_span_days":      grp["timestamp"].apply(lambda x: (x.max() - x.min()).days),
        "mean_word_count":         grp["word_count"].mean(),
        "mean_mention_count":      grp["mention_count"].mean(),
        "emoji_usage_rate":        grp["has_emoji"].mean(),
        "question_rate":           grp["has_question"].mean(),
        "exclamation_rate":        grp["has_exclaim"].mean(),
    }
    if "engagement_rate" in df.columns:
        agg["mean_post_engagement_rate"] = grp["engagement_rate"].mean()
    feat = pd.DataFrame(agg).reset_index()
    feat["post_concentration_ratio"] = (
        feat["unique_posts_commented"] / feat["total_comments"]
    ).clip(upper=1.0)
    return feat

user_features = build_user_feature_matrix(ig_comments)
print(f"\u2705 User feature matrix built: {len(user_features):,} users.")

# 4. Attach representative comment text from comments_llm.jsonl
llm_path = f"gs://{GCS_BUCKET}/comments_llm_instagram.jsonl"
print(f"Loading comment text from {llm_path} ...")
ig_comments_text = pd.read_json(llm_path, lines=True)

top_comments = (
    ig_comments_text.groupby("author_id").head(5)
    .groupby("author_id")["text"]
    .apply(lambda ts: " ||| ".join(ts.astype(str).tolist()))
    .reset_index()
    .rename(columns={"text": "top_comments_sample"})
)
user_features = user_features.merge(top_comments, on="author_id", how="left")
print("\u2705 Representative comment history attached.")
display(user_features.head(3))

## Stage 1 — Taxonomy Discovery

Sends batches of user profiles to Gemini Flash to identify latent behavioral
archetypes. Output is saved to `outputs/taxonomy.json` for human review.

**Workflow:**
1. Run this section (`PIPELINE_MODE = "SAMPLE"`)
2. Open `outputs/taxonomy.json`, consolidate candidates into a MECE taxonomy
3. Set `"status": "APPROVED"` in the JSON
4. Proceed to Stage 2

### Vertex AI Context Caching Helper

In [ ]:
from vertexai.preview import caching
import datetime

def create_explicit_vertex_cache(
    system_instruction: str,
    context_text: str,
    model_name: str = MODEL_STAGE2_CLASSIFY,
):
    """Pre-computes a server-side KV cache to avoid re-tokenizing the taxonomy
    system prompt on every Stage 2 request."""
    print("Pre-computing Vertex AI Context Cache (TTL: 1h)...")
    cached = caching.CachedContent.create(
        model_name=model_name,
        system_instruction=system_instruction,
        contents=[context_text],
        ttl=datetime.timedelta(hours=1),
    )
    print(f"\u2705 Cache created: {cached.name}")
    return cached

print("\u2705 Context caching helper ready.")

### Stage 1 Functions

In [ ]:
import json, re, time, os
from tqdm import tqdm

STAGE1_SYSTEM_PROMPT = 'You are an expert community analyst for a major Italian influencer agency.\nYour task is to identify distinct audience persona archetypes from user comment behaviour data.\n\nYou will receive a batch of Instagram commenter profiles. Each profile contains:\n- Quantitative behavioral metrics (comment frequency, timing, engagement)\n- A sample of their actual comments (pipe-separated)\n\nYour goal is to identify RECURRING behavioral patterns across these users.\nFor each pattern you observe, output a candidate persona with:\n- A short codename (e.g., "SUPERFAN", "BRAND_ADVOCATE")\n- A 1-sentence behavioral description\n- Key distinguishing quantitative signals\n- 2-3 representative verbatim comment fragments\n\nOutput ONLY a valid JSON array of persona objects. No preamble, no markdown fences.\nSchema: [{"codename": str, "description": str, "signals": [str], "examples": [str]}]'


def format_user_profile_for_stage1(row: pd.Series) -> str:
    return (
        f"USER: {row['author_id']} | "
        f"Total comments: {int(row['total_comments'])} | "
        f"Unique posts: {int(row['unique_posts_commented'])} | "
        f"Activity span: {int(row['activity_span_days'])} days | "
        f"Avg hrs to comment: {row['mean_hours_to_comment']:.1f}h | "
        f"Early commenter (<1h): {row['pct_comments_under_1h']:.0%} | "
        f"Reply ratio: {row['reply_ratio']:.0%} | "
        f"Avg word count: {row['mean_word_count']:.0f} | "
        f"Emoji rate: {row['emoji_usage_rate']:.0%} | "
        f"Question rate: {row['question_rate']:.0%} | "
        f"Sample comments: {str(row.get('top_comments_sample', ''))[:400]}"
    )


def run_stage1_taxonomy_discovery(
    user_features_df: pd.DataFrame,
    n_sample: int = SAMPLE_N_USERS,
    batch_size: int = BATCH_SIZE_STAGE1,
    seed: int = SAMPLE_SEED,
) -> list[dict]:
    print(f"\n{'='*60}")
    print(f"STAGE 1 — Exploratory Taxonomy Discovery")
    print(f"  Sample: {n_sample} users | Batch size: {batch_size}")
    print(f"  Model: {MODEL_STAGE1_EXPLORATORY}")
    print(f"{'='*60}\n")

    sample_df = user_features_df.sample(
        n=min(n_sample, len(user_features_df)), random_state=seed
    ).reset_index(drop=True)

    model   = get_model(MODEL_STAGE1_EXPLORATORY)
    config  = get_generation_config(MAX_OUTPUT_TOKENS_STAGE1)
    batches = [sample_df.iloc[i:i+batch_size] for i in range(0, len(sample_df), batch_size)]

    all_candidates = []
    for batch_idx, batch in enumerate(tqdm(batches, desc="Stage 1 batches")):
        profiles_text = "\n\n".join(
            format_user_profile_for_stage1(row) for _, row in batch.iterrows()
        )
        prompt = (
            f"{STAGE1_SYSTEM_PROMPT}\n\n"
            f"--- USER PROFILES BATCH {batch_idx + 1} of {len(batches)} ---\n\n"
            f"{profiles_text}\n\n"
            "Identify all distinct behavioral archetypes present in this batch.\n"
            "Output ONLY the JSON array."
        )
        try:
            raw = model.generate_content(prompt, generation_config=config).text.strip()
            raw = re.sub(r"^```json\s*", "", raw)
            raw = re.sub(r"\s*```$", "", raw)
            candidates = json.loads(raw)
            if isinstance(candidates, list):
                all_candidates.extend(candidates)
            print(f"   Batch {batch_idx+1}: {len(candidates)} candidates found.")
        except Exception as e:
            print(f"   \u26a0\ufe0f  Batch {batch_idx+1} error: {e}")
        time.sleep(1.5)

    print(f"\n\u2705 Stage 1 complete. Total raw candidates: {len(all_candidates)}")
    return all_candidates


def save_taxonomy_for_review(candidates: list[dict], output_path: str = TAXONOMY_JSON_PATH):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    output = {
        "status": "PENDING_HUMAN_REVIEW",
        "instructions": (
            "Review and consolidate the candidates below into a MECE taxonomy. "
            "Each final persona must have a unique 'codename'. "
            "Change status to 'APPROVED' before running Stage 2."
        ),
        "raw_candidates": candidates,
        "final_taxonomy": [],
    }
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    print(f"\u2705 Raw candidates saved to: {output_path}")
    print("   \u26a0\ufe0f  HUMAN ACTION REQUIRED: Review, consolidate, set status='APPROVED'.")


print("\u2705 Stage 1 functions ready.")

### Stage 1 — Inspect Taxonomy Results

In [ ]:
if os.path.exists(TAXONOMY_JSON_PATH):
    with open(TAXONOMY_JSON_PATH, "r", encoding="utf-8") as f:
        taxonomy_data = json.load(f)
    status = taxonomy_data.get("status")
    print(f"Taxonomy status: {status}")
    if status == "APPROVED" and taxonomy_data.get("final_taxonomy"):
        display(pd.DataFrame(taxonomy_data["final_taxonomy"]))
    elif taxonomy_data.get("raw_candidates"):
        print("Showing raw candidates (pending review):")
        display(pd.DataFrame(taxonomy_data["raw_candidates"]))
    else:
        print("No candidates found — run Stage 1 first.")
else:
    print("taxonomy.json not found — run Stage 1 first.")

## Stage 2 — Deterministic Classification

Classifies every user in the dataset into exactly one persona from the approved taxonomy.

**CAG architecture:**
- The taxonomy is compiled once into a static system prompt (the cache layer)
- User profiles are sent in small batches as the variable user turn
- Checkpoints save every 10 batches to support scheduled runtime restarts

In [ ]:
def build_stage2_system_prompt(taxonomy: list[dict]) -> str:
    taxonomy_text = ""
    for p in taxonomy:
        taxonomy_text += (
            f"\nPERSONA: {p['codename']} — {p['label']}\n"
            f"  Description: {p['description']}\n"
            f"  Quantitative signals: {'; '.join(p.get('quantitative_signals', []))}\n"
            f"  Example comments: {' | '.join(p.get('example_comments', []))}\n"
        )
    return (
        "You are a deterministic community analyst for Show Reel Media Group.\n"
        "Your task is to classify Instagram commenters into exactly ONE persona from the approved taxonomy.\n\n"
        "=== APPROVED PERSONA TAXONOMY ===\n"
        f"{taxonomy_text}"
        "=================================\n\n"
        "CLASSIFICATION RULES:\n"
        "1. Each user MUST be assigned to exactly ONE persona — the closest match.\n"
        "2. You MUST output a confidence score between 0.0 and 1.0.\n"
        "3. You MUST cite a specific text fragment from the user's comments as justification.\n"
        "4. If a user's data is insufficient, assign the most probable persona with confidence ≤ 0.4.\n"
        "5. Output ONLY a valid JSON array. No preamble, no markdown fences.\n\n"
        "JSON Output Schema per user:\n"
        "{\n  \"author_id\": str,\n  \"persona_codename\": str,\n  \"confidence\": float,\n  \"justification\": str\n}"
    )


def format_user_profile_for_stage2(row: pd.Series) -> dict:
    return {
        "author_id":               str(row["author_id"]),
        "total_comments":          int(row["total_comments"]),
        "unique_posts":            int(row["unique_posts_commented"]),
        "activity_span_days":      int(row["activity_span_days"]),
        "pct_comments_under_1h":   round(float(row["pct_comments_under_1h"]), 2),
        "pct_comments_under_24h":  round(float(row["pct_comments_under_24h"]), 2),
        "reply_ratio":             round(float(row["reply_ratio"]), 2),
        "mean_mention_count":      round(float(row["mean_mention_count"]), 2),
        "mean_word_count":         round(float(row["mean_word_count"]), 1),
        "emoji_usage_rate":        round(float(row["emoji_usage_rate"]), 2),
        "question_rate":           round(float(row["question_rate"]), 2),
        "exclamation_rate":        round(float(row["exclamation_rate"]), 2),
        "post_concentration_ratio": round(float(row["post_concentration_ratio"]), 2),
        "sample_comments":         str(row.get("top_comments_sample", ""))[:500],
    }


def run_stage2_classification(
    user_features_df: pd.DataFrame,
    taxonomy: list[dict],
    ig_media_df: pd.DataFrame,
    batch_size: int = BATCH_SIZE_STAGE2,
    output_path: str = RESULTS_PATH,
    resume: bool = True,
) -> pd.DataFrame:
    print(f"\n{'='*60}")
    print("STAGE 2 — Deterministic CAG Classification")
    print(f"  Total users: {len(user_features_df):,} | Batch size: {batch_size}")
    print(f"  Model: {MODEL_STAGE2_CLASSIFY} | Temp: {TEMPERATURE}")
    print(f"{'='*60}\n")

    checkpoint_path = output_path.replace(".parquet", "_checkpoint.parquet")
    if resume and os.path.exists(checkpoint_path):
        done_df  = pd.read_parquet(checkpoint_path)
        done_ids = set(done_df["author_id"].astype(str))
        todo_df  = user_features_df[~user_features_df["author_id"].isin(done_ids)]
        results  = done_df.to_dict("records")
        print(f"   Resuming: {len(done_ids):,} done, {len(todo_df):,} remaining.")
    else:
        todo_df = user_features_df.copy()
        results = []

    system_prompt = build_stage2_system_prompt(taxonomy)
    model   = get_model(MODEL_STAGE2_CLASSIFY)
    config  = get_generation_config(MAX_OUTPUT_TOKENS_STAGE2)
    batches = [todo_df.iloc[i:i+batch_size] for i in range(0, len(todo_df), batch_size)]
    error_count = 0

    for batch_idx, batch in enumerate(tqdm(batches, desc="Stage 2 classification")):
        user_profiles   = [format_user_profile_for_stage2(row) for _, row in batch.iterrows()]
        user_batch_text = json.dumps(user_profiles, ensure_ascii=False, indent=2)
        full_prompt = (
            f"{system_prompt}\n\n"
            f"=== USER PROFILES TO CLASSIFY (Batch {batch_idx + 1} of {len(batches)}) ===\n"
            f"{user_batch_text}\n\n"
            "Classify each user into exactly one persona. Output the JSON array only."
        )
        try:
            raw = model.generate_content(full_prompt, generation_config=config).text.strip()
            raw = re.sub(r"^```json\s*", "", raw)
            raw = re.sub(r"\s*```$", "", raw)
            classified = json.loads(raw)
            if isinstance(classified, list):
                results.extend(classified)
            if (batch_idx + 1) % 10 == 0:
                pd.DataFrame(results).to_parquet(checkpoint_path, index=False)
                print(f"   Checkpoint saved at batch {batch_idx + 1}.")
        except Exception as e:
            error_count += 1
            print(f"   \u26a0\ufe0f  Batch {batch_idx+1} error: {e}")
            for profile in user_profiles:
                results.append({
                    "author_id": profile["author_id"],
                    "persona_codename": "CLASSIFICATION_ERROR",
                    "confidence": 0.0,
                    "justification": f"API error: {str(e)[:100]}",
                })
        time.sleep(1.0)

    results_df  = pd.DataFrame(results)
    summary_cols = ["author_id", "total_comments", "activity_span_days",
                    "mean_hours_to_comment", "pct_comments_under_1h",
                    "reply_ratio", "mean_word_count"]
    final_df = user_features_df[summary_cols].merge(results_df, on="author_id", how="left")
    final_df.to_parquet(output_path, index=False)

    print(f"\n\u2705 Stage 2 complete.")
    print(f"   Classified: {len(final_df):,} users | Errors: {error_count} batches")
    print(f"   Output: {output_path}")

    dist = final_df["persona_codename"].value_counts(normalize=True).mul(100).round(2)
    print("\n   Persona Distribution:")
    for persona, pct in dist.items():
        print(f"      {persona:<35} {pct:.1f}%")

    return final_df


print("\u2705 Stage 2 functions ready.")

## Pipeline Orchestrator

Single entry point for the Colab Enterprise scheduler.  
Set `PIPELINE_MODE` in the Configuration cell, then run this cell.

In [ ]:
def load_approved_taxonomy(path: str = TAXONOMY_JSON_PATH) -> list[dict]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if data.get("status") != "APPROVED":
        raise ValueError(
            f"Taxonomy at '{path}' is not approved. "
            "Complete human-in-the-loop review and set status='APPROVED'."
        )
    taxonomy = data.get("final_taxonomy", [])
    if not taxonomy:
        raise ValueError("final_taxonomy is empty.")
    print(f"\u2705 Approved taxonomy loaded: {len(taxonomy)} personas.")
    return taxonomy


def run_pipeline(mode: str = PIPELINE_MODE):
    print(f"\n{'#'*60}")
    print(f"  SHOW REEL PERSONA PIPELINE — MODE: {mode}")
    print(f"{'#'*60}\n")

    if mode in ("SAMPLE", "ALL"):
        candidates = run_stage1_taxonomy_discovery(
            user_features_df=user_features,
            n_sample=SAMPLE_N_USERS,
            batch_size=BATCH_SIZE_STAGE1,
            seed=SAMPLE_SEED,
        )
        save_taxonomy_for_review(candidates, TAXONOMY_JSON_PATH)
        if mode == "SAMPLE":
            print("\n\u23f8  Paused after Stage 1.")
            print("   ACTION: Review and approve taxonomy at:", TAXONOMY_JSON_PATH)
            return None

    if mode in ("FULL", "ALL"):
        taxonomy = load_approved_taxonomy(TAXONOMY_JSON_PATH)
        return run_stage2_classification(
            user_features_df=user_features,
            taxonomy=taxonomy,
            ig_media_df=ig_media,
            batch_size=BATCH_SIZE_STAGE2,
            output_path=RESULTS_PATH,
            resume=True,
        )

    return None


result = run_pipeline(mode=PIPELINE_MODE)

## Upload Outputs to GCS

Run after the pipeline completes to persist results across runtimes.

In [ ]:
def upload_outputs_to_gcs(local_dir: str = "outputs/"):
    from google.cloud import storage
    client = storage.Client(project=GCP_PROJECT_ID)
    bucket = client.bucket(GCS_BUCKET)
    for fname in os.listdir(local_dir):
        local_path = os.path.join(local_dir, fname)
        blob       = bucket.blob(fname)
        blob.upload_from_filename(local_path)
        print(f"   Uploaded: {local_path} → gs://{GCS_BUCKET}/{fname}")

# upload_outputs_to_gcs()

## Validation & QA

Quick post-classification health checks:
- Coverage rate (% of users successfully classified)
- Confidence distribution
- MECE check (no user classified twice)
- Low-confidence flagging (< 0.4)

In [ ]:
def validate_classification_output(results_path: str = RESULTS_PATH):
    df = pd.read_parquet(results_path)

    classified   = df["persona_codename"].notna() & (df["persona_codename"] != "CLASSIFICATION_ERROR")
    coverage_pct = classified.sum() / len(df) * 100

    print("\nCLASSIFICATION QA REPORT")
    print(f"   Total users             : {len(df):,}")
    print(f"   Successfully classified : {classified.sum():,} ({coverage_pct:.1f}%)")
    print(f"   Errors                  : {(~classified).sum():,}")

    sub = df[classified].copy()
    sub["confidence"] = pd.to_numeric(sub["confidence"], errors="coerce")
    print(f"\n   Confidence  mean: {sub['confidence'].mean():.3f}  "
          f"median: {sub['confidence'].median():.3f}  "
          f"<0.40: {(sub['confidence'] < 0.4).sum():,}")

    dupes = df["author_id"].duplicated().sum()
    print(f"\n   MECE Check: {dupes} duplicate assignments "
          f"({'FAIL' if dupes > 0 else 'PASS'})")

    display(df["persona_codename"].value_counts().to_frame("count"))
    return df


# qa_df = validate_classification_output()